In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

# Load the dataset
file_path = 'C:/Users/nosao/Desktop/Maxwell-Text Classification/Matrix Response/data/Matrix_Teaching_learning_support.csv' #Replace with actual file path
df = pd.read_csv(file_path)

# Extract job descriptions and target variables for Questions 19, 20, 21
X = df['Job description']
y_q46 = df['Question 46']
y_q47 = df['Question 47']
y_q48 = df['Question 48']
y_q49 = df['Question 49']



# Split data into training and test sets for each question
X_train, X_test, y_train_q46, y_test_q46 = train_test_split(X, y_q46, test_size=0.2, random_state=42)
X_train_q47, X_test_q47, y_train_q47, y_test_q47 = train_test_split(X, y_q47, test_size=0.2, random_state=42)
X_train_q48, X_test_q48, y_train_q48, y_test_q48 = train_test_split(X, y_q48, test_size=0.2, random_state=42)
X_train_q49, X_test_q49, y_train_q49, y_test_q49 = train_test_split(X, y_q49, test_size=0.2, random_state=42)

# Define the model and pipeline for Logistic Regression
logreg_pipeline = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))

# Hyperparameter tuning grid
param_grid_logreg = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidfvectorizer__max_df': [0.85, 0.9, 0.95],
    'tfidfvectorizer__min_df': [1, 5],
    'tfidfvectorizer__use_idf': [True, False],
    'tfidfvectorizer__sublinear_tf': [True, False]
}

# Hyperparameter tuning for Logistic Regression for Question 46
grid_logreg_q46 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=5, scoring='accuracy', n_jobs=-1)
grid_logreg_q46.fit(X_train, y_train_q46)

# Hyperparameter tuning for Logistic Regression for Question 47
grid_logreg_q47 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=5, scoring='accuracy', n_jobs=-1)
grid_logreg_q47.fit(X_train_q47, y_train_q47)

# Hyperparameter tuning for Logistic Regression for Question 48
grid_logreg_q48 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=5, scoring='accuracy', n_jobs=-1)
grid_logreg_q48.fit(X_train_q48, y_train_q48)

# Hyperparameter tuning for Logistic Regression for Question 49
grid_logreg_q49 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=5, scoring='accuracy', n_jobs=-1)
grid_logreg_q49.fit(X_train_q49, y_train_q49)

# Use the best Logistic Regression models for each question
best_logreg_model_q46 = grid_logreg_q46.best_estimator_
best_logreg_model_q47 = grid_logreg_q47.best_estimator_
best_logreg_model_q48 = grid_logreg_q48.best_estimator_
best_logreg_model_q49 = grid_logreg_q49.best_estimator_

# Function to upload new job descriptions and compare predicted and real responses for all questions
def predict_job_description(new_job_desc, true_response_q46=None, true_response_q47=None, true_response_q48=None, true_response_q49=None):
    # Predict the response for each question
    predicted_response_q46 = best_logreg_model_q46.predict([new_job_desc])[0]
    predicted_response_q47 = best_logreg_model_q47.predict([new_job_desc])[0]
    predicted_response_q48 = best_logreg_model_q48.predict([new_job_desc])[0]
    predicted_response_q49 = best_logreg_model_q49.predict([new_job_desc])[0]

    # Display the results
    print(f"Job Description: {new_job_desc}")

    # Question 46
    print(f"Predicted Response for Question 46: {predicted_response_q46}")

    # Question 47
    print(f"Predicted Response for Question 47: {predicted_response_q47}")

    # Question 48
    print(f"Predicted Response for Question 48: {predicted_response_q48}")

    # Question 49
    print(f"Predicted Response for Question 49: {predicted_response_q49}")


    return predicted_response_q46, predicted_response_q47, predicted_response_q48, predicted_response_q49

# Function to display metrics (accuracy, precision, recall, F1-score)
def display_metrics(y_true, y_pred, question_num):
    print(f"Metrics for Question {question_num}:")
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}")
    print(classification_report(y_true, y_pred))

# Predict on the full test set for each question and display metrics
y_pred_q46 = best_logreg_model_q46.predict(X_test)
display_metrics(y_test_q46, y_pred_q46, 46)

y_pred_q47 = best_logreg_model_q47.predict(X_test_q47)
display_metrics(y_test_q47, y_pred_q47, 47)

y_pred_q48 = best_logreg_model_q48.predict(X_test_q48)
display_metrics(y_test_q48, y_pred_q48, 48)

y_pred_q49 = best_logreg_model_q49.predict(X_test_q49)
display_metrics(y_test_q49, y_pred_q49, 49)

# Function to make predictions on a new job description
def predict_for_new_job_description(job_description):
    # Ensure the input is a string
    job_description = [job_description]

    # Predict for each question using the best model
    pred_q46 = best_logreg_model_q46.predict(job_description)[0]
    pred_q47 = best_logreg_model_q47.predict(job_description)[0]
    pred_q48 = best_logreg_model_q48.predict(job_description)[0]
    pred_q49 = best_logreg_model_q49.predict(job_description)[0]

    # Display or return the results
    print("Predictions for the new job description:")
    print(f"Question 12: {pred_q46}")
    print(f"Question 13: {pred_q47}")
    print(f"Question 14: {pred_q48}")
    print(f"Question 15: {pred_q49}")

# Example: Enter a new job description
new_job_description = """
Purpose of the Role: To provide an effective Joinery resource to ensure the University
fabric is efficiently maintained on a day-to-day basis including undertaking Project works. To
ensure the effective interaction of Estate and Facilities services with other services.

Responsible to: Estates Team Leader

Main Duties and Responsibilities:
1. To provide all forms of Joinery duties and tasks in which you are competent within the
University Estate possessing at least five years of trade experience. Working with the team
across various other construction trades.
2. To be responsible for day-to-day breakdown and reactive maintenance.
3. To participate in the Maintenance call-out rota team.
4. To be responsible for working to and delivering cyclical maintenance works ensuring
certain activities are carried out as per the PPM regime.
5. To manage the fire door programme focussing on the Inspection, maintenance (ART
accepted repair techniques), and repair to achieve a compliant campus. This will involve
a good theoretical knowledge of the standards and regulations.
6. To be responsible for the Fire door dashboard, upkeep, and maintenance of the system.
To provide information on defects and repair (analyzing reports), accepted repair
techniques, and input for Projects.
7. To maintain Fire door records on CAFM and in line with the Fire Safety Regulations
(2023). Records must be kept.
8. Identify hazards, defects, and the need for adjustment or repair; to ensure compliance
with agreed codes, law, working practices, and health and safety whilst carrying out your
duties.
9. To provide support and guidance to Contractors engaged in Fire door campus works and
act as a focal point ensuring a fully compliant Fire door install is delivered to the Estate.
10. To manage quantities required to complete each task and manage material stocks and
ordering process.
11. To be responsible for ensuring all tools and equipment are maintained in good working
order and ready for use including power tools within the Estate.
12. To ensure all University fixtures, fittings, furniture, doors, locks, flooring, and other
Joinery items are efficiently maintained, repaired, constructed, or replaced, working
closely with Estates, Facilities Managers, and Estates Team Leader to achieve.
13. To ensure works are delivered in compliance with documented risk assessments and Method
statements, and responsible for the production and review of role-specific risk
assessments.
14. To be responsible for a high standard of conduct always working in a safe and
professional manner reporting any health and safety-related issues to the Estates Team
Leader immediately.
15. To be responsible for working to and delivering all works and repairs in a manner that
ensures VFM and quality finishes are implemented and maintained.
16. To assist the team and organization with general duties over and above your core skills.
Promote, develop and expand the business of our organization generally meeting set
targets.
17. To aid and advise the other members of the University Estates & Facilities staff including
porters and grounds staff as required.
18. To adhere to all organization policies and procedures.
19. To be responsible for continued professional development ensuring the post holder is
conversant and aware of current regulations, legislation, and approved industry
standards to their job role. You will have a basic awareness of Asbestos, CDM,
health and safety regulations.
20. To undertake small projects as reasonably required of the job role and advise all Estates
teams to deliver solutions and cost reductions to all joinery works on campus.

General Duties:
21. To ensure the use of data complies with current regulations, particularly those relating to
GDPR.
22. To comply with all health, safety, and wellbeing policies and procedures at all times and to
take responsibility for promoting and safeguarding the welfare and protection of others.
23. To advocate, promote, and advance equity and social justice within your work.
24. To carry out other duties, commensurate with the grade of the post, as may reasonably be
directed by your line manager after due consultation.
"""

# Call the function with the job description
predict_for_new_job_description(new_job_description)

# Create a folder for saving models
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)  # Create the folder 

# Save the models in the "saved_models" directory
joblib.dump(best_logreg_model_q46, os.path.join(save_dir, "model_q46.pkl"))
joblib.dump(best_logreg_model_q47, os.path.join(save_dir, "model_q47.pkl"))
joblib.dump(best_logreg_model_q48, os.path.join(save_dir, "model_q48.pkl"))
joblib.dump(best_logreg_model_q49, os.path.join(save_dir, "model_q49.pkl"))

print("Models saved in the 'saved_models' folder.")



c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Metrics for Question 46:
Accuracy: 0.8
              precision    recall  f1-score   support

           C       0.86      0.67      0.75         9
           D       0.77      0.91      0.83        11

    accuracy                           0.80        20
   macro avg       0.81      0.79      0.79        20
weighted avg       0.81      0.80      0.80        20

Metrics for Question 47:
Accuracy: 0.75
              precision    recall  f1-score   support

           B       0.00      0.00      0.00         3
           C       0.50      0.67      0.57         3
           D       0.81      0.93      0.87        14

    accuracy                           0.75        20
   macro avg       0.44      0.53      0.48        20
weighted avg       0.64      0.75      0.69        20

Metrics for Question 48:
Accuracy: 0.85
              precision    recall  f1-score   support

           A       0.00      0.00      0.00         2
           B       1.00      0.50      0.67         2
          

c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [ ]:
# Example: Enter a new job description
new_job_description = """

"""

# Call the function with the job description
predict_for_new_job_description(new_job_description)

Predictions for the new job description:
Question 12: D
Question 13: D
Question 14: D
Question 15: D


In [2]:
# Define the saved models directory
save_dir = "saved_models"

# Load models correctly
model_q46 = joblib.load(os.path.join(save_dir, "model_q46.pkl"))
model_q47 = joblib.load(os.path.join(save_dir, "model_q47.pkl"))
model_q48 = joblib.load(os.path.join(save_dir, "model_q48.pkl"))
model_q49 = joblib.load(os.path.join(save_dir, "model_q49.pkl"))

print("Models loaded successfully!")

Models loaded successfully!
